# Danh: S2/RCTC với MotionBlurAug của bạn — đối chứng đúng, Kaggle 2×T4

**Mục tiêu:** Kiểm xem augmentation trong notebook `aug-svtrv2.ipynb` có giúp **đúng vùng train Stage 2 của Danh** hay không. Hai nhánh cùng bắt đầu từ checkpoint SVTRv2-T chính chủ, **chỉ train Stage 2 + Stage 3 + RCTC**, cùng Union14M-L-Filter medium (144.677 ảnh trong notebook bạn), 12 epoch, batch 48, LR và seed cho mô hình/sampler. Nhánh `S2_NOAUG` vẫn có `PARSeqAugPIL` như pipeline ảnh train gốc của OpenOCR; tên NOAUG ở đây chỉ nghĩa **không có MotionBlurAug**. Nhánh `S2_AUG` chèn nguyên thuật toán `MotionBlurAug` của notebook bạn sau `PARSeqAugPIL`. MotionBlurAug gốc dùng PID/thời gian cho RNG nên không thể tái hiện từng ảnh đã augment chỉ từ seed. Hai GPU chạy hai nhánh độc lập song song.

**Nguồn:** [SVTRv2/OpenOCR](https://github.com/Topdu/OpenOCR) cung cấp checkpoint, mô hình và vòng train; thuật toán MotionBlurAug được lấy từ file notebook bạn cung cấp, là code của nhóm, không phải hàm gốc từ paper. Mở Stage 2 + Stage 3 + RCTC là ablation S2 trước của Danh. Notebook của bạn báo 80,23% → 89,70% blur nặng trên một cấu hình khác: train toàn mô hình GTC+RCTC 38.304 bước. Số đó **không chứng minh** S2/RCTC cũng đạt 89,70%.

**Khác với pilot S2 300 bước:** pilot đó dùng loss CTC đã gộp chữ hoa/thường và bỏ dấu câu. Notebook mới dùng `CTCLabelEncode`/`CTCLoss` chính thức của OpenOCR để train Union14M như notebook bạn; vì vậy xem đây là **S2-Union**, không lấy chênh lệch với 82,30% pilot để gán riêng cho augmentation. Phép so có thể kết luận về augmentation là `S2_AUG` **so với** `S2_NOAUG` trong chính notebook này.

**Lưu ý phép đo:** augment được hiệu chỉnh bằng manifest của bộ IIIT5K blur, vì thế đây là đánh giá trên một protocol đã biết. Nhóm đã xem test nhiều lần, cần test mới để khẳng định tổng quát. Dữ liệu Union14M lớn cũng chưa được kiểm trùng ảnh ở mức nội dung với IIIT5K test. Không dùng test để chọn checkpoint; đánh giá checkpoint **cuối epoch 12** cho cả hai nhánh.


## 0. Input và cấu hình

Trên Kaggle bật **T4×2**, Internet và gắn dataset blur có `degradation_manifest.csv`, `blur/`. Nếu dùng đường dẫn khác, sửa `BLUR_DATA_HINT` trong cell. Cell cài đặt, đọc dataset, tạo LMDB và `MotionBlurAug` bên dưới được mang từ notebook `aug-svtrv2.ipynb`; khác biệt của thử nghiệm S2 nằm ở các cell tiếp theo.


In [ ]:
BLUR_DATA_HINT='/kaggle/input/datasets/nguynthnhthy/data-blur'
LOCAL_TINY_CKPT=''  # optional /kaggle/input/.../best.pth if Drive quota blocks download
LOCAL_TINY_CFG=''   # optional /kaggle/input/.../config.yml
BATCH,ACCUM=48,1
TRAIN_PREF=['medium','normal']; MAX_TRAIN_GB=8
TRAIN_LMDB_OVERRIDE=[]
EXTRA_KINDS=[]
SEED=1234
RESTORED_CAM=False
EXP=dict(family='svtrv2',aug=True,aug_p=.5,len_scale=1.,angle_mode='protocol',epochs=12,lr=2e-4,warmup_frac=.075)
SV_VARIANT='T'
print('Fixed pair: S2_NOAUG vs S2_AUG; only MotionBlurAug differs')


In [ ]:
import os, sys, io, re, glob, time, json, random, subprocess, shutil, zipfile, tarfile
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image
WORK = Path('/kaggle/working/s2_unionaug_2026_v1') if Path('/kaggle').exists() else Path('work/s2_unionaug_2026_v1'); WORK.mkdir(exist_ok=True, parents=True)
OPENOCR = WORK/'OpenOCR'
if not OPENOCR.exists():
    subprocess.run(['git','clone','https://github.com/Topdu/OpenOCR.git',str(OPENOCR)], check=True)
subprocess.run(['git','-C',str(OPENOCR),'checkout','1ccfc6ee6161f7133b192e16af3a9265273997ba'],check=True)
assert subprocess.check_output(['git','-C',str(OPENOCR),'rev-parse','HEAD'],text=True).strip()=='1ccfc6ee6161f7133b192e16af3a9265273997ba'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(OPENOCR/'requirements.txt')], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','--only-binary',':all:','pygame'], check=False)   # KHÔNG ghim 2.5.2: không có wheel cho Python mới
subprocess.run([sys.executable,'-m','pip','install','-q','lmdb','gdown','huggingface_hub','scipy','tabulate'], check=False)
# ---- VÁ tương thích (Python 3.13 / numpy 2 / pygame 2.6 trên Kaggle) ----
import importlib.util
_spec = importlib.util.find_spec('imgaug')            # không import imgaug (đang hỏng với numpy 2)
if _spec:
    _root = Path(_spec.origin).parent
    _rep = [(r'np\.sctypes\["float"\]', '[np.float16, np.float32, np.float64]'),
            (r'np\.sctypes\["int"\]',   '[np.int8, np.int16, np.int32, np.int64]'),
            (r'np\.sctypes\["uint"\]',  '[np.uint8, np.uint16, np.uint32, np.uint64]'),
            (r'np\.complex(?![_0-9A-Za-z])', 'np.complex128'), (r'np\.bool(?![_0-9A-Za-z])', 'np.bool_'),
            (r'np\.float(?![_0-9A-Za-z])', 'np.float64'),     (r'np\.int(?![_0-9A-Za-z])', 'np.int64')]
    _n = 0
    for _f in _root.rglob('*.py'):
        _s = _f.read_text(); _t = _s
        for _a, _b in _rep: _t = re.sub(_a, _b, _t)
        if _t != _s: _f.write_text(_t); _n += 1
    print('vá imgaug:', _n, 'file')
_p = OPENOCR/'openrec/preprocess/cam_label_encode.py'
_s = _p.read_text(); _p.write_text(_s.replace('pygame.locals.SRCALPHA', 'pygame.SRCALPHA')); print('vá cam_label_encode (pygame.SRCALPHA)')
# kiểm tra các import mà train/eval của OpenOCR cần, trong tiến trình con (đúng môi trường lúc train)
for mod in ['pygame.freetype','imgaug','lmdb','cv2','rapidfuzz','pyclipper']:
    r = subprocess.run([sys.executable,'-c',f'import {mod}'], capture_output=True, text=True)
    print(('OK   ' if r.returncode==0 else 'LỖI ')+mod, '' if r.returncode==0 else r.stderr.strip().splitlines()[-1])
    assert r.returncode == 0, f'Thiếu/hỏng thư viện {mod}: xem dòng LỖI ở trên.'
os.chdir(OPENOCR); sys.path.insert(0, str(OPENOCR))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(torch.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Bật GPU trong Settings > Accelerator.'
# CAM vẽ mask glyph bằng font; bản gốc dùng Arial. Không có thì dùng Liberation Sans (GHI RÕ trong báo cáo). Chỉ cần cho họ CAM.
FONT_USED = None
cands_f = glob.glob('/usr/share/fonts/**/arial.ttf', recursive=True) or glob.glob('/usr/share/fonts/**/LiberationSans-Regular.ttf', recursive=True)
if cands_f: shutil.copy(cands_f[0], OPENOCR/'arial.ttf'); FONT_USED = cands_f[0]
if EXP['family'] == 'cam': assert cands_f, 'Không có font. Tải arial.ttf đặt vào thư mục OpenOCR/.'
print('font mask:', FONT_USED)

# vá torch.load trong OpenOCR: torch>=2.6 mặc định weights_only=True làm hỏng nạp/resume checkpoint
_npt = 0
for _f in list(OPENOCR.glob('tools/**/*.py')) + list(OPENOCR.glob('openrec/**/*.py')):
    _s = _f.read_text(); _t = re.sub(r'torch\.load\(([^()\n]*?)\)', lambda m: m.group(0) if 'weights_only' in m.group(1) else 'torch.load(' + m.group(1) + ', weights_only=False)', _s)
    if _t != _s: _f.write_text(_t); _npt += 1
print('vá torch.load(weights_only=False):', _npt, 'file')


In [ ]:
assert torch.cuda.device_count()==2, 'Chọn Kaggle T4 x2 để chạy hai nhánh song song'
print('GPU', [torch.cuda.get_device_name(i) for i in range(2)])

## 1. Dựng đúng bộ test và dữ liệu train của notebook bạn

Phần này lấy ảnh blur dựng sẵn, kiểm thứ tự ảnh/nhãn, và lấy Union14M-L-Filter medium. Bản gốc và hai nhánh S2 sẽ đọc **cùng LMDB test**. Bước hiệu chỉnh augment xem thông số kernel trong manifest, chỉ để khớp protocol, không đưa ảnh test vào train.


In [ ]:
def find_blur_root():
    if BLUR_DATA_HINT:
        hit = glob.glob(f'{BLUR_DATA_HINT}/**/degradation_manifest*', recursive=True)
        assert hit, f'Không thấy degradation_manifest* dưới {BLUR_DATA_HINT}'
        return Path(hit[0]).parent
    for base in ['/kaggle/input', '/kaggle/working', '.']:
        hit = glob.glob(f'{base}/**/degradation_manifest*', recursive=True)
        if hit: return Path(hit[0]).parent
    for base in ['/kaggle/input', '/kaggle/working', '.']:
        z = glob.glob(f'{base}/**/IIIT5K_Blur_Data*.zip', recursive=True)
        if z:
            out = WORK/'blur_data'
            if not out.exists(): zipfile.ZipFile(z[0]).extractall(out)
            hit = glob.glob(f'{out}/**/degradation_manifest*', recursive=True)
            if hit: return Path(hit[0]).parent
    raise FileNotFoundError('Không thấy dữ liệu blur. Add dataset vào notebook hoặc đặt BLUR_DATA_HINT.')
BROOT = find_blur_root(); print('dữ liệu blur:', BROOT)
man_path = sorted(glob.glob(str(BROOT/'degradation_manifest*')))[0]
man = pd.read_csv(man_path) if man_path.endswith('.csv') else pd.read_excel(man_path)
print(man_path, man.shape); print(list(man.columns)); display(man.head(6))

In [ ]:
# Tự đoán cột theo tên; SỬA tay nếu in ra sai.
def pick(keys):
    for c in man.columns:
        if any(k in c.lower() for k in keys): return c
    return None
COLS = dict(file=pick(['file','path','image','name']), label=pick(['label','text','gt','word','transcr']),
            kind=pick(['type','kind','blur','degrad']), level=pick(['level','sever','strength','intens']))
print('COLS =', COLS)
for k in ('kind','level'):
    if COLS[k]: print(k, '->', man[COLS[k]].astype(str).value_counts().to_dict())

In [ ]:
import lmdb, scipy.io
IMG_EXT = {'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}
imgs = {p.name: p for p in (BROOT/'blur').rglob('*') if p.suffix.lower() in IMG_EXT}
print('số ảnh blur:', len(imgs))
def img_id(name):
    m = re.search(r'iiit_test_(\d+)', name); return int(m.group(1)) if m else None
def tail_of(name):
    t = name.split('__', 1)[1] if '__' in name else ''; return os.path.splitext(t)[0]
tails = {}
for n in imgs: tails[tail_of(n)] = tails.get(tail_of(n), 0) + 1
print('các điều kiện blur (phần đuôi tên file) và số ảnh:')
for t, c in sorted(tails.items()): print(f'   {t!r}: {c}')

def kind_of(s):
    s = s.lower()
    if 'gauss' in s: return 'gaussian'
    if any(w in s for w in ['defocus','disk','circ','lens','bokeh']): return 'defocus'
    if any(w in s for w in ['motion','linear']): return 'motion'
    if any(w in s for w in ['clean','orig']): return 'clean'
    return None
def level_of(s):
    s = s.lower()
    for k, v in [('mild','mild'),('med','medium'),('sev','severe'),('serve','severe'),('strong','severe'),('low','mild'),('mid','medium'),('high','severe')]:
        if k in s: return v
    return None
kinds = {t: kind_of(t) for t in tails}
unk = [t for t in tails if kinds[t] is None]
assert not unk, f'Không nhận diện được loại blur cho các đuôi: {unk}. Gửi danh sách ở trên để chỉnh kind_of.'
tail_cond = {}
for k in set(kinds.values()):
    ts = [t for t in tails if kinds[t] == k]; lv = {t: level_of(t) for t in ts}
    if k == 'clean': tail_cond.update({t: 'clean' for t in ts}); continue
    if None in lv.values() or len(set(lv.values())) != len(ts):
        assert len(ts) == 3, f'{k}: {len(ts)} đuôi, không gán được mild/medium/severe: {ts}'
        order = sorted(ts, key=lambda t: [int(x) if x.isdigit() else x for x in re.split(r'(\d+)', t)])
        lv = dict(zip(order, ['mild','medium','severe']))
        print('CẢNH BÁO: không đọc được mức từ tên file; gán theo thứ tự tự nhiên', k, lv, '-> KIỂM TRA lại với tham số blur thật')
    tail_cond.update({t: f'{k}_{lv[t]}' for t in ts})
print('ánh xạ đuôi -> điều kiện:', tail_cond)

# ---- ảnh sạch + nhãn: lấy từ nguồn IIIT5K (thư mục blur/ không có ảnh sạch) ----
def read_lmdb_pairs(path):
    env = lmdb.open(str(path), readonly=True, lock=False, readahead=False); out = []
    with env.begin() as t:
        total = int(t.get(b'num-samples')); base = 1 if t.get(b'image-%09d' % 1) is not None else 0
        for i in range(base, base+total):
            ib, lb = t.get(b'image-%09d' % i), t.get(b'label-%09d' % i)
            out.append((Image.open(io.BytesIO(ib)).convert('RGB'), lb.decode('utf-8','ignore')))
    env.close(); return out

def load_clean_source():
    roots = [BROOT, BROOT.parent] + ([Path(BLUR_DATA_HINT)] if BLUR_DATA_HINT else [])
    mats = [m for r in roots for m in r.rglob('testdata.mat')]
    if not mats:
        tars = [p for r in roots for p in r.rglob('*') if p.is_file() and re.search(r'\.tar(\.\w+)?$', p.name)]
        print('tar tìm thấy:', tars)
        if tars:
            ex = WORK/'iiit5k_src'
            if not ex.exists(): tarfile.open(tars[0]).extractall(ex)
            mats = list(ex.rglob('testdata.mat'))
    if mats:
        mat = mats[0]; print('nguồn sạch: testdata.mat ở', mat)
        td = scipy.io.loadmat(mat, squeeze_me=True, struct_as_record=False)['testdata']
        out = []
        for nm, gt in zip(td.ImgName, td.GroundTruth):
            p = mat.parent/str(nm)
            if not p.exists(): p = mat.parent/'test'/os.path.basename(str(nm))
            out.append((Image.open(p).convert('RGB'), str(gt)))
        return out
    print('không có tar/mat trong dataset; dùng IIIT5k LMDB từ Hugging Face (topdu/OpenOCR-Data/evaluation/IIIT5k)')
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='topdu/OpenOCR-Data', repo_type='dataset', local_dir=WORK/'hf_eval', allow_patterns=['evaluation/IIIT5k/*'])
    return read_lmdb_pairs(WORK/'hf_eval'/'evaluation'/'IIIT5k')

clean_src = load_clean_source(); print('ảnh sạch nguồn:', len(clean_src))
CLEAN_DIR = WORK/'clean_png'; CLEAN_DIR.mkdir(exist_ok=True)
clean_paths = []
for i, (im, lb) in enumerate(clean_src):
    p = CLEAN_DIR/f'iiit_test_{i:05d}__clean.png'
    if not p.exists(): im.save(p)
    clean_paths.append(p)

# ---- kiểm tra: id ảnh blur có khớp thứ tự ảnh sạch không? (đối sánh ảnh mờ nhẹ nhất với ảnh sạch) ----
def feat(p):
    a = np.asarray(Image.open(p).convert('L').resize((64, 16), Image.BILINEAR), np.float32).ravel()
    a = a - a.mean(); return a / (np.linalg.norm(a) + 1e-6)
chk = sorted([c for c in set(tail_cond.values()) if c != 'clean'], key=lambda c: (not c.endswith('mild'), c))[0]
chk_tail = [t for t, c in tail_cond.items() if c == chk][0]
chk_items = sorted([(img_id(n), p) for n, p in imgs.items() if tail_of(n) == chk_tail])
ids = [i for i, _ in chk_items]; assert max(ids) < len(clean_paths), f'id lớn nhất {max(ids)} >= số ảnh sạch {len(clean_paths)}'
Xb = np.stack([feat(p) for _, p in chk_items]); Xc = np.stack([feat(p) for p in clean_paths])
nn_idx = (Xb @ Xc.T).argmax(1); agree = float(np.mean(nn_idx == np.array(ids)))
print(f'kiểm tra khớp id ({chk}): {agree*100:.1f}% ảnh mờ giống nhất đúng ảnh sạch cùng chỉ số')
if agree >= 0.95: id2lab = {i: clean_src[i][1] for i in range(len(clean_src))}
else:
    print('CẢNH BÁO: id không khớp thứ tự nguồn -> lấy nhãn theo ảnh sạch giống nhất (đối sánh)')
    id2lab = {i: clean_src[j][1] for i, j in zip(ids, nn_idx)}
assert agree > 0.5, 'Ảnh blur và ảnh nguồn gần như không tương ứng; kiểm tra lại nguồn IIIT5K.'

GROUPS = {'clean': [(clean_paths[i], clean_src[i][1]) for i in sorted(id2lab)]}
for n, p in imgs.items():
    c = tail_cond[tail_of(n)]
    if c != 'clean': GROUPS.setdefault(c, []).append((p, id2lab.get(img_id(n))))
print({c: len(v) for c, v in sorted(GROUPS.items())})
assert all(f'motion_{l}' in GROUPS for l in ['mild','medium','severe']), 'Thiếu motion_mild/medium/severe: xem ánh xạ đuôi -> điều kiện ở trên.'
assert all(lb is not None for v in GROUPS.values() for _, lb in v), 'Có ảnh không có nhãn.'

In [ ]:
def write_lmdb(items, out):
    Path(out).mkdir(parents=True, exist_ok=True); env = lmdb.open(str(out), map_size=1<<33); n = 0
    with env.begin(write=True) as t:
        for p, lb in sorted(items, key=lambda x: x[0].name):
            try: im = Image.open(p).convert('RGB')
            except Exception: continue
            n += 1; b = io.BytesIO(); im.save(b, 'PNG')
            t.put(b'image-%09d' % n, b.getvalue()); t.put(b'label-%09d' % n, lb.encode())
        t.put(b'num-samples', str(n).encode())
    env.close(); return n
LM = WORK/'blur_lmdb'; N = {}
for cond, items in GROUPS.items():
    d = LM/cond/'test'
    N[cond] = int(open(d/'n.txt').read()) if (d/'n.txt').exists() else write_lmdb(items, d)
    if not (d/'n.txt').exists(): open(d/'n.txt','w').write(str(N[cond]))
print(N)
CONDS = ['clean'] + [f'motion_{l}' for l in ['mild','medium','severe']] + [f'{k}_{l}' for k in EXTRA_KINDS for l in ['mild','medium','severe'] if f'{k}_{l}' in GROUPS]
print('sẽ đánh giá:', CONDS)

In [ ]:
TRAIN_ROOT = WORK/'train_root'
picked = []
if RESTORED_CAM:
    print('đã khôi phục checkpoint CAM -> bỏ qua tải dữ liệu train')
else:
    if TRAIN_LMDB_OVERRIDE:
        picked = [Path(p) for p in TRAIN_LMDB_OVERRIDE]
    else:
        from huggingface_hub import HfApi, snapshot_download
        REPO = 'topdu/OpenOCR-Data'; api = HfApi()
        files = api.list_repo_files(REPO, repo_type='dataset')
        mdb = [f for f in files if f.endswith('data.mdb')]
        tl = [f for f in mdb if 'train' in f.lower() and not f.lower().startswith(('evaluation','u14m'))]
        print('thư mục LMDB train tìm thấy:', [os.path.dirname(f) for f in tl][:20])
        assert tl, 'Repo HF không có data.mdb train. Điền TRAIN_LMDB_OVERRIDE hoặc build LMDB theo docs/svtrv2.md.'
        info = {i.path: i.size for i in api.get_paths_info(REPO, tl, repo_type='dataset')}
        ok = {f: s_ for f, s_ in info.items() if s_ and s_/1e9 <= MAX_TRAIN_GB}
        assert ok, f'Mọi thư mục train > {MAX_TRAIN_GB} GB: {({os.path.dirname(f): round(s_/1e9,1) for f, s_ in info.items()})}'
        def rank(f):
            for i, k in enumerate(TRAIN_PREF):
                if k in f.lower(): return (i, ok[f])
            return (len(TRAIN_PREF), ok[f])
        best = sorted(ok, key=rank)[0]; d = os.path.dirname(best)
        print('chọn:', d, f'({ok[best]/1e9:.2f} GB)')
        if not (Path(WORK/'hf_data')/d/'data.mdb').exists():
            snapshot_download(repo_id=REPO, repo_type='dataset', local_dir=WORK/'hf_data', allow_patterns=[f'{d}/*'])
        picked = [WORK/'hf_data'/d]
    # OpenOCR quét thư mục train bằng os.walk (KHÔNG theo symlink) -> dùng hardlink/copy thật
    import lmdb
    shutil.rmtree(TRAIN_ROOT, ignore_errors=True); TRAIN_ROOT.mkdir()
    for i, p in enumerate(picked):
        dst = TRAIN_ROOT/f'part{i}'; dst.mkdir()
        for f in Path(p).iterdir():
            if f.is_file():
                try: os.link(f, dst/f.name)
                except OSError: shutil.copy2(f, dst/f.name)
    leaf = [d_ for d_, dn, fn in os.walk(str(TRAIN_ROOT) + '/') if not dn]
    tot = 0
    for d_ in leaf:
        env = lmdb.open(d_, readonly=True, lock=False); tot += int(env.begin().get(b'num-samples')); env.close()
    print('thư mục LMDB train:', leaf, '| số mẫu train:', tot)
    assert tot >= BATCH, 'Không thấy dữ liệu train.'

In [ ]:
assert len(picked)==1 and 'filter_train_medium' in str(picked[0]) and tot==144677, 'Train corpus differs from medium 144677; audit before continuing'
print('TRAIN CORPUS AUDIT PASS',picked,tot)

In [ ]:
RESF = WORK/'results.json'
RES = json.load(open(RESF)) if RESF.exists() else {}
def save(): json.dump(RES, open(RESF,'w'), indent=1)

def run_eval(cfg, ckpt, cond, kind):
    lm = LM/cond
    key = f'Eval.dataset.data_dir_list=[{lm}/test]' if kind == 'svtrv2' else f'Eval.dataset.data_dir={lm}'
    cmd = [sys.executable, 'tools/eval_rec.py', '--c', cfg, '--o', f'Global.pretrained_model={ckpt}', key,
           'Eval.loader.batch_size_per_card=128', 'Global.use_amp=False']
    r = subprocess.run(cmd, cwd=OPENOCR, capture_output=True, text=True); out = r.stdout + r.stderr
    m = re.findall(r'\bacc[:=]\s*([0-9.]+)', out)
    if not m: print(out[-2000:]); raise RuntimeError(f'Không đọc được acc ({cond}).')
    return 100*float(m[-1])

def eval_all(name, cfg, ckpt, kind):
    RES.setdefault(name, {})
    for c in CONDS:
        if c not in RES[name]: RES[name][c] = run_eval(cfg, ckpt, c, kind); save(); print(name, c, round(RES[name][c], 2))
    return RES[name]
def wilson(acc, n, z=1.96):
    p = acc/100; d = 1+z*z/n; c = p+z*z/(2*n); h = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n)); return 100*(c-h)/d, 100*(c+h)/d

In [ ]:
# No cache across notebook versions/checkpoints: every reported score is computed here.
RES={}
def eval_all(name,cfg,ckpt,kind):
 RES[name]={}
 for cond in CONDS:
  RES[name][cond]=run_eval(cfg,ckpt,cond,kind)
  print(name,cond,round(RES[name][cond],3))
 return RES[name]


In [ ]:
# ===== Dữ liệu tham chiếu của OpenOCR: IIIT5k gốc (kiểm tra baseline đọc đúng ~98%) + IC13 (chỉ để THEO DÕI lúc train, không chọn checkpoint) =====
from huggingface_hub import snapshot_download
snapshot_download(repo_id='topdu/OpenOCR-Data', repo_type='dataset', local_dir=WORK/'hf_eval', allow_patterns=['evaluation/IIIT5k/*', 'evaluation/IC13_857/*'])
IIIT_ORI = str(WORK/'hf_eval/evaluation/IIIT5k')
assert (Path(IIIT_ORI)/'data.mdb').exists(), 'Không tải được IIIT5k gốc từ HF (bật Internet trong Settings của Kaggle).'
VAL_DIR = WORK/'hf_eval/evaluation/IC13_857'
if not (VAL_DIR/'data.mdb').exists():
    VAL_DIR = Path(IIIT_ORI); print('CẢNH BÁO: không có IC13_857, theo dõi trên IIIT5k gốc (chỉ để theo dõi, tuyệt đối không chọn epoch/checkpoint theo số này)')
print('IIIT5k gốc:', IIIT_ORI, '| tập theo dõi:', VAL_DIR)


## 2. Checkpoint SVTRv2-T chính chủ và augmentation gốc của bạn

Không tự chọn checkpoint/config theo accuracy test. Dùng đúng hai file release của tác giả bằng ID và SHA256 đã audit ở các lượt S2 trước. Đo B0 trên chính LMDB test này để phát hiện khác biệt protocol với các lượt trước.


In [ ]:
import hashlib,gdown,yaml,copy
CK=WORK/'official_tiny.pth';SV_CFG=WORK/'official_config.yml'
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
for path,gid,expect in [(CK,'1xJ10gOQphZjEgawXsgsadkXaZa0BIDN7','684f23bc3a2b1f724cadfd275e1a185456dd51b5e56c8830780d1998e90f77f6'),
 (SV_CFG,'1a-NRSh9aoYpdtwNgctF7a5CbEHcTA71a','f1ab35a0f5e44d3c24c940cdf1a07404068ad9191bad9fe232ad4fcf38bbbe29')]:
 local=LOCAL_TINY_CKPT if path==CK else LOCAL_TINY_CFG
 if local and not path.exists():shutil.copyfile(local,path)
 if not path.exists():gdown.download(id=gid,output=str(path),quiet=False)
 assert path.exists() and sha(path)==expect,f'Wrong official file {path}'
SV_CKPT=str(CK);SV_CFG=str(SV_CFG)
display(pd.DataFrame({'file':[SV_CKPT,SV_CFG],'sha256':[sha(CK),sha(SV_CFG)]}))
base_scores=eval_all('B0_official',SV_CFG,SV_CKPT,'svtrv2')
print('B0 on this dataset:',{k:round(v,2) for k,v in base_scores.items()})
assert abs(base_scores['clean']-98.60)<.5 and abs(base_scores['motion_severe']-80.23)<.5,   'Baseline differs from friend notebook; audit test data/config before long training'


In [ ]:
# ===== Tham số motion blur của nhóm -> thông số tăng cường (chỉ tổng hợp blur lên ẢNH TRAIN; không dùng ảnh test để train) =====
MBA_SRC = r'''import math
import os
import time

import cv2
import numpy as np
from PIL import Image


class MotionBlurAug:
    # Làm mờ chuyển động tuyến tính cho ảnh train (PIL RGB). Độ dài kernel tỉ lệ với chiều cao ảnh
    # để sau khi resize về 32/64 px độ mờ nhìn giống nhau với mọi cỡ ảnh gốc.

    def __init__(self, p=0.5, len_frac=(0.1, 0.5), angle=(0.0, 180.0), min_len=3, **kwargs):
        self.p = float(p)
        self.len_frac = (float(len_frac[0]), float(len_frac[1]))
        self.angle = (float(angle[0]), float(angle[1]))
        self.min_len = int(min_len)
        self._rng = None
        self._pid = None

    def _get_rng(self):
        if self._rng is None or self._pid != os.getpid():        # mỗi worker của DataLoader có luồng ngẫu nhiên riêng
            self._pid = os.getpid()
            self._rng = np.random.default_rng([self._pid, time.time_ns() % (2**32)])
        return self._rng

    @staticmethod
    def kernel(length, angle_deg):
        k = max(3, int(round(length)))
        if k % 2 == 0:
            k += 1
        ker = np.zeros((k, k), np.float32)
        c = k // 2
        a = math.radians(angle_deg)
        dx, dy = math.cos(a) * (k - 1) / 2.0, math.sin(a) * (k - 1) / 2.0
        cv2.line(ker, (int(round(c - dx)), int(round(c - dy))), (int(round(c + dx)), int(round(c + dy))), 1.0, 1)
        s = float(ker.sum())
        if s <= 0:
            ker[c, c] = 1.0
            s = 1.0
        return ker / s

    def blur(self, arr, length, angle_deg):
        return cv2.filter2D(arr, -1, self.kernel(length, angle_deg), borderType=cv2.BORDER_REPLICATE)

    def __call__(self, data):
        rng = self._get_rng()
        if rng.random() >= self.p:
            return data
        arr = np.asarray(data['image'])
        length = rng.uniform(*self.len_frac) * arr.shape[0]
        if length < self.min_len:
            return data
        data['image'] = Image.fromarray(self.blur(arr, length, rng.uniform(*self.angle)))
        return data
'''
(OPENOCR/'openrec/preprocess/motion_blur_aug.py').write_text(MBA_SRC)
_p = OPENOCR/'openrec/preprocess/__init__.py'; _s = _p.read_text()
if "'MotionBlurAug'" not in _s: _p.write_text(_s.replace('MODULE_MAPPING = {', "MODULE_MAPPING = {\n    'MotionBlurAug': '.motion_blur_aug',", 1))
import cv2, importlib
importlib.invalidate_caches()
from openrec.preprocess.motion_blur_aug import MotionBlurAug

def _num(v):
    try: return float(v)
    except Exception: return None

def derive_motion_params(man, COLS, GROUPS, id_of, n_sample=300, n_img=120):
    '''Đọc tham số motion blur thật trong manifest -> (độ dài/chiều cao ảnh, góc) theo từng mức. Trả về dict hoặc None nếu không đọc được.'''
    kc, lc = COLS['kind'], COLS['level']
    info = {}
    for lvl in ['mild', 'medium', 'severe']:
        rows = man[(man[kc].astype(str).str.lower() == 'motion') & (man[lc].astype(str).str.lower() == lvl)]
        if len(rows) == 0: return None
        rows = rows.sample(min(n_sample, len(rows)), random_state=0)
        vals = {}
        for s in rows['blur_parameters']:
            try: d = json.loads(s) if isinstance(s, str) else dict(s)
            except Exception: continue
            for k, v in d.items():
                x = _num(v)
                if x is not None: vals.setdefault(k, []).append(x)
        info[lvl] = {k: float(np.median(v)) for k, v in vals.items() if v}
        info[lvl]['_raw'] = vals
        hs = [Image.open(p).size[1] for p, _ in GROUPS[f'motion_{lvl}'][:n_img]]
        info[lvl]['_h'] = float(np.median(hs))
    keys = [k for k in info['mild'] if not k.startswith('_')]
    print('khóa tham số motion trong manifest:', keys)
    for lvl in info: print('  ', lvl, {k: round(v, 3) for k, v in info[lvl].items() if not k.startswith('_')}, '| chiều cao ảnh trung vị', info[lvl]['_h'])
    def pick(words, prefer='actual'):
        c = [k for k in keys if any(w in k.lower() for w in words)]
        c.sort(key=lambda k: (not k.lower().startswith(prefer), len(k))); return c[0] if c else None
    lk = pick(['length', 'kernel', 'extent', 'blur_len', 'size']); ak = pick(['angle', 'theta', 'direction'])
    if lk is None: return None
    frac = {l: info[l][lk] / info[l]['_h'] for l in info}
    out = dict(len_key=lk, angle_key=ak, len_frac_by_level=frac)
    if ak:
        a = np.concatenate([info[l]['_raw'][ak] for l in info]); deg = a if np.abs(a).max() > 6.5 else np.degrees(a)
        out['angle_obs'] = (float(np.percentile(deg, 1)), float(np.percentile(deg, 99)))
    return out

def calibration_check(params, GROUPS, id_of, n=150):
    '''So độ mờ (tỉ lệ phương sai Laplacian mờ/sạch) giữa blur THẬT của nhóm và blur giả lập của MotionBlurAug trên cùng ảnh sạch.'''
    def lv(p):
        g = cv2.cvtColor(np.asarray(Image.open(p).convert('RGB')), cv2.COLOR_RGB2GRAY); return float(cv2.Laplacian(g, cv2.CV_64F).var()) + 1e-9
    clean = {id_of(p.name): p for p, _ in GROUPS['clean']}
    rng = np.random.default_rng(0); aug = MotionBlurAug(p=1.0); res = {}
    for lvl in ['mild', 'medium', 'severe']:
        real, syn = [], []
        for p, _ in GROUPS[f'motion_{lvl}'][:n]:
            c = clean.get(id_of(p.name))
            if c is None: continue
            im = np.asarray(Image.open(c).convert('RGB')); h = im.shape[0]
            g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY); l0 = float(cv2.Laplacian(g, cv2.CV_64F).var()) + 1e-9
            real.append(lv(p) / l0)
            ang = rng.uniform(*params['angle']) ; b = aug.blur(im, params['len_frac_by_level'][lvl] * h, ang)
            syn.append(float(cv2.Laplacian(cv2.cvtColor(b, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()) / l0)
        res[lvl] = (float(np.median(real)), float(np.median(syn)))
        flag = '' if abs(res[lvl][1]/res[lvl][0] - 1) < 0.35 else '   <-- LỆCH >35%: chỉnh LEN_SCALE trong EXP'
        print(f'  {lvl}: độ sắc còn lại (Laplacian) thật = {res[lvl][0]:.3f} | giả lập = {res[lvl][1]:.3f}{flag}')
    return res

def id_of(name):
    m = re.search(r'iiit_test_(\d+)', name); return int(m.group(1)) if m else name

mp = None
try: mp = derive_motion_params(man, COLS, GROUPS, id_of)
except Exception as e: print('không đọc được tham số motion:', repr(e))
if mp is None:
    print('CẢNH BÁO: không suy ra được độ dài/góc blur từ manifest -> dùng dải RỘNG mặc định (0.05-0.6 chiều cao, 0-180 độ).'
          ' Cần kiểm tra dòng "khóa tham số motion" ở trên để chỉnh cho khớp protocol.')
    LEN_FRAC = (0.05, 0.6); ANGLE = (0.0, 180.0); MP_SOURCE = 'mặc định rộng (không khớp protocol)'
else:
    fr = list(mp['len_frac_by_level'].values()); LEN_FRAC = (0.6*min(fr)*EXP['len_scale'], 1.2*max(fr)*EXP['len_scale'])
    ANGLE = (0.0, 180.0) if (EXP['angle_mode'] == 'any' or 'angle_obs' not in mp) else (mp['angle_obs'][0]-10, mp['angle_obs'][1]+10)
    MP_SOURCE = f"manifest (khóa độ dài '{mp['len_key']}', góc '{mp['angle_key']}')"
    print('độ dài/chiều cao theo mức:', {k: round(v, 3) for k, v in mp['len_frac_by_level'].items()})
    try: calibration_check(dict(angle=ANGLE, len_frac_by_level={k: v*EXP['len_scale'] for k, v in mp['len_frac_by_level'].items()}), GROUPS, id_of)
    except Exception as e: print('bỏ qua kiểm tra hiệu chỉnh:', repr(e))
AUG_OP = {'MotionBlurAug': dict(p=EXP['aug_p'], len_frac=list(LEN_FRAC), angle=list(ANGLE))}
print('MotionBlurAug dùng:', AUG_OP, '| nguồn:', MP_SOURCE, '| bật:', EXP['aug'])

In [ ]:
# ===== Trực quan PHA DỮ LIỆU: ảnh sạch | blur THẬT của nhóm (3 mức) | blur GIẢ LẬP của MotionBlurAug (3 mức) | glyph mask (họ CAM) =====
import matplotlib.pyplot as plt
def show_data_phase(n=4):
    clean = {id_of(p.name): (p, lb) for p, lb in GROUPS['clean']}
    real = {l: {id_of(p.name): p for p, _ in GROUPS[f'motion_{l}']} for l in ['mild', 'medium', 'severe']}
    ids = sorted(set(clean) & set(real['mild']) & set(real['medium']) & set(real['severe'])); step = max(1, len(ids)//n); ids = ids[::step][:n]
    fr = (mp['len_frac_by_level'] if mp else dict(zip(['mild', 'medium', 'severe'], np.linspace(LEN_FRAC[0], LEN_FRAC[1], 3))))
    ang = float(np.mean(ANGLE)); aug = MotionBlurAug(p=1.0)
    cols = ['sạch'] + [f'thật-{l}' for l in ['mild', 'medium', 'severe']] + [f'giả lập-{l}' for l in ['mild', 'medium', 'severe']] + (['glyph mask (từ nhãn)'] if EXP['family'] == 'cam' else [])
    fig, axs = plt.subplots(len(ids), len(cols), figsize=(2.3*len(cols), 1.5*len(ids)+0.8), squeeze=False)
    for r, i in enumerate(ids):
        p, lb = clean[i]; im = np.asarray(Image.open(p).convert('RGB')); ims = [im] + [np.asarray(Image.open(real[l][i]).convert('RGB')) for l in ['mild', 'medium', 'severe']]
        ims += [aug.blur(im, fr[l]*EXP['len_scale']*im.shape[0], ang) for l in ['mild', 'medium', 'severe']]
        if EXP['family'] == 'cam':
            from PIL import ImageDraw, ImageFont
            g = Image.new('L', (256, 64), 0); f = ImageFont.truetype(FONT_USED, 44); ImageDraw.Draw(g).text((6, 4), lb, fill=255, font=f); ims.append(np.asarray(g))
        for c_, a in enumerate(ims):
            ax = axs[r][c_]; ax.imshow(a, cmap=None if a.ndim == 3 else 'gray'); ax.axis('off')
            if r == 0: ax.set_title(cols[c_], fontsize=9)
    plt.suptitle('Pha dữ liệu: minh họa blur trên test; khi train MotionBlurAug chỉ áp lên ảnh TRAIN', fontsize=10); plt.tight_layout(); plt.savefig(WORK/'data_phase.png', dpi=120); plt.show()
try: show_data_phase()
except Exception as e: print('bỏ qua hình minh họa pha dữ liệu:', repr(e)[:200])


In [ ]:
assert (WORK/'data_phase.png').exists(), 'Không tạo được hình kiểm dữ liệu; dừng trước train'

## 3. Chuyển checkpoint gốc sang đúng đầu RCTC của S2

Release Tiny có GTC + RCTC. S2 trước dùng RCTC để đọc chữ. Ở đây chỉ lấy trọng số encoder và nhánh RCTC, bỏ nhánh GTC; kiểm `strict=True` và số tham số **5.128.255**. Hai nhánh đều nhận cùng checkpoint vừa chuyển.


In [ ]:
from openrec.modeling import build_model
raw=torch.load(SV_CKPT,map_location='cpu',weights_only=False)
state={k.removeprefix('module.'):v for k,v in raw['state_dict'].items()}
mapped={};dropped=[]
for k,v in state.items():
 if k.startswith('encoder.'):mapped[k]=v
 elif k.startswith('decoder.ctc_decoder.'):mapped['decoder.'+k[len('decoder.ctc_decoder.'):]]=v
 elif k.startswith('decoder.gtc_decoder.'):dropped.append(k)
 else:raise ValueError('Unexpected checkpoint key: '+k)
arch=copy.deepcopy(yaml.safe_load(open(SV_CFG))['Architecture'])
assert arch['Encoder']['dims']==[64,128,256] and arch.get('Transform') is None
arch['algorithm']='SVTRv2';arch['Decoder']={'name':'RCTCDecoder','out_channels':mapped['decoder.fc.weight'].shape[0]}
model=build_model(arch);model.load_state_dict(mapped,strict=True)
assert sum(p.numel() for p in model.parameters())==5128255
S2_INIT=WORK/'s2_rctc_official_init.pth'
torch.save({'state_dict':model.state_dict()},S2_INIT)
print('S2 RCTC strict load OK; GTC keys removed',len(dropped),'init SHA',sha(S2_INIT))
del model,raw,state;torch.cuda.empty_cache()


## 4. Hai cấu hình chỉ khác MotionBlurAug

`S2_NOAUG`: `Decode → PARSeqAugPIL → CTCLabelEncode`.

`S2_AUG`: `Decode → PARSeqAugPIL → MotionBlurAug → CTCLabelEncode`.

Các yếu tố còn lại, kể cả checkpoint đầu, tập ảnh, lịch học và vùng được train, giữ như nhau. Đây là đối chứng cần có mà notebook của bạn chưa có: so với B0 thôi chưa tách được lợi ích augmentation khỏi lợi ích của train thêm 12 epoch.


In [ ]:
RCTC_REF=yaml.safe_load(open(OPENOCR/'configs/rec/svtrv2/svtrv2_tiny_rctc.yml'))
BASE_CFG=yaml.safe_load(open(SV_CFG))
def leaf_dirs(root):
 return [d for d,sub,files in os.walk(str(root)+'/') if not sub and (Path(d)/'data.mdb').exists()]
TRAIN_DIRS=[str(p) for p in leaf_dirs(TRAIN_ROOT)]
assert TRAIN_DIRS and all(Path(x,'data.mdb').exists() for x in TRAIN_DIRS)
def make_cfg(name,with_aug):
 c=copy.deepcopy(BASE_CFG)
 c['Global'].update(pretrained_model=str(S2_INIT),checkpoints=None,output_dir=str(WORK/name),
   epoch_num=EXP['epochs'],seed=SEED,accumulation_steps=1,cal_metric_during_train=False,
   eval_epoch_step=[0,1],eval_batch_step=[10**12,500],save_epoch_step=[0,1],use_amp=True)
 c['Optimizer']['lr']=EXP['lr'];c['LRScheduler']['warmup_epoch']=round(EXP['warmup_frac']*EXP['epochs'],3)
 c['Architecture']=copy.deepcopy(arch);c['Architecture']['Decoder'].pop('out_channels')
 for key in ('Loss','PostProcess','Metric'):c[key]=copy.deepcopy(RCTC_REF[key])
 tr=[{'DecodeImagePIL':{'img_mode':'RGB'}},{'PARSeqAugPIL':None}]
 if with_aug:tr.append(copy.deepcopy(AUG_OP))
 tr += [{'CTCLabelEncode':{'character_dict_path':str(OPENOCR/'tools/utils/EN_symbol_dict.txt'),
          'use_space_char':False,'max_text_length':25}},
        {'KeepKeys':{'keep_keys':['image','label','length']}}]
 c['Train']['dataset']['transforms']=tr
 c['Train']['dataset']['data_dir_list']=TRAIN_DIRS
 c['Train']['dataset'].pop('ratio_list',None)
 c['Train']['sampler']['first_bs']=48
 c['Train']['loader']['batch_size_per_card']=48
 c['Train']['loader']['num_workers']=2
 c['Eval']['dataset']['transforms']=[tr[0],tr[-2],tr[-1]]
 c['Eval']['dataset']['data_dir_list']=[str(VAL_DIR)]
 c['Eval']['loader']['batch_size_per_card']=128
 c['Eval']['loader']['num_workers']=2
 dst=WORK/f'{name}.yml';yaml.safe_dump(c,open(dst,'w'),sort_keys=False)
 return dst,c
NOAUG_CFG,noaug=make_cfg('S2_NOAUG',False)
AUG_CFG,withaug=make_cfg('S2_AUG',True)
diff=[]
for section in ('Architecture','Loss','PostProcess','Metric','Optimizer','LRScheduler','Train','Eval'):
 a=copy.deepcopy(noaug[section]);b=copy.deepcopy(withaug[section])
 if section=='Train':b['dataset']['transforms']=[v for v in b['dataset']['transforms'] if 'MotionBlurAug' not in v]
 if a!=b:diff.append(section)
assert not diff,diff
assert noaug['Global']['pretrained_model']==withaug['Global']['pretrained_model']
assert [list(x)[0] for x in withaug['Train']['dataset']['transforms']]==['DecodeImagePIL','PARSeqAugPIL','MotionBlurAug','CTCLabelEncode','KeepKeys']
print('CONFIG AUDIT PASS: only MotionBlurAug differs; train LMDB',TRAIN_DIRS)


## 5. S2 trainable audit và smoke test trên GPU

Script dùng OpenOCR `Trainer` nguyên vòng train; chỉ override bước dựng mô hình để đóng băng phần đầu **trước khi tạo optimizer**. Khóa BatchNorm thống kê như S2 cũ. `--smoke` chạy một batch, kiểm Stage 1 không có gradient, Stage 2/3 và RCTC có gradient, loss hữu hạn. Nếu sai, dừng trước lượt train dài.


In [ ]:
from matplotlib.patches import FancyBboxPatch
fig,ax=plt.subplots(figsize=(13,2.5));ax.set(xlim=(0,13),ylim=(0,2.5));ax.axis('off')
labels=[('Ảnh train',.1,'#dbeafe'),('Aug tùy nhánh',2.0,'#ffedd5'),('Patch',4.1,'#e5e7eb'),
 ('Stage 1',5.45,'#e5e7eb'),('Stage 2',6.85,'#dcfce7'),('Stage 3',8.25,'#dcfce7'),
 ('RCTC',9.65,'#dcfce7'),('Chữ',11.05,'#e0e7ff')]
for label,x,color in labels:
 ax.add_patch(FancyBboxPatch((x,1.1),1.2,.65,boxstyle='round,pad=.06',facecolor=color,edgecolor='#456'))
 ax.text(x+.6,1.42,label,ha='center',va='center',fontsize=9)
for (_,x,_),(_,nx,_) in zip(labels,labels[1:]):
 ax.annotate('',xy=(nx,1.42),xytext=(x+1.2,1.42),arrowprops=dict(arrowstyle='->',lw=1.2))
ax.text(.2,.35,'Cam: MotionBlurAug chỉ ở nhánh AUG khi train. Xanh: train Stage 2 + Stage 3 + RCTC ở cả hai nhánh. Khi test không dùng Aug.',fontsize=9)
plt.tight_layout();plt.savefig(WORK/'S2_AUG_pipeline.png',dpi=150);plt.show()


In [ ]:
S2_SCRIPT=r'''import argparse,sys,torch,json
from pathlib import Path
sys.path.insert(0,str(Path(__file__).resolve().parent/'OpenOCR'))
from tools.engine.config import Config
from tools.engine.trainer import Trainer

class S2Trainer(Trainer):
 def _init_rec_model(self):
  super()._init_rec_model()
  m=self.model
  assert len(m.encoder.stages)==4 and not tuple(m.encoder.stages[3].parameters())
  m.requires_grad_(False)
  m.encoder.stages[1].requires_grad_(True)
  m.encoder.stages[2].requires_grad_(True)
  m.decoder.requires_grad_(True)
  assert not any(p.requires_grad for p in m.encoder.pope.parameters())
  assert not any(p.requires_grad for p in m.encoder.stages[0].parameters())
  assert all(p.requires_grad for p in m.encoder.stages[1].parameters())
  assert all(p.requires_grad for p in m.encoder.stages[2].parameters())
  assert all(p.requires_grad for p in m.decoder.parameters())
  # Trainer.train()/eval_step() toggles mode; keep BN statistics fixed as in prior S2.
  def keep_bn(module,inputs):
   for sub in module.modules():
    if isinstance(sub,torch.nn.modules.batchnorm._BatchNorm):sub.eval()
  m.register_forward_pre_hook(keep_bn)

def main():
 ap=argparse.ArgumentParser();ap.add_argument('--config',required=True);ap.add_argument('--smoke',action='store_true');a=ap.parse_args()
 cfg=Config(a.config)
 out=Path(cfg.cfg['Global']['output_dir']);latest=out/'latest.pth'
 if latest.exists():
  saved=torch.load(latest,map_location='cpu',weights_only=False)
  if saved['epoch']>=cfg.cfg['Global']['epoch_num']:
   print('COMPLETE',latest,'epoch',saved['epoch']);return
  cfg.cfg['Global']['checkpoints']=str(latest)
 t=S2Trainer(cfg,mode='train_eval',task='rec');m=t.model
 if cfg.cfg['Global']['checkpoints'] is None:
  expected=torch.load(cfg.cfg['Global']['pretrained_model'],map_location='cpu',weights_only=False)['state_dict']
  actual=m.state_dict();assert set(actual)==set(expected)
  assert all(torch.equal(actual[k].detach().cpu(),v) for k,v in expected.items()),'Initial weights differ from converted official checkpoint'
 names=[n for n,p in m.named_parameters() if p.requires_grad]
 print('S2 TRAINABLE',len(names),'parameters',sum(p.numel() for p in m.parameters() if p.requires_grad))
 if a.smoke:
  b=next(iter(t.train_dataloader));tensor=[x.to(t.device) for x in b]
  m.train();pred=m(tensor[0],data=tensor[1:]);loss=t.loss_class(pred,tensor)['loss']
  assert bool(torch.isfinite(loss));loss.backward()
  assert any(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in m.encoder.stages[1].parameters())
  assert any(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in m.encoder.stages[2].parameters())
  assert any(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in m.decoder.parameters())
  assert all(p.grad is None for p in m.encoder.stages[0].parameters())
  (out/'smoke.json').write_text(json.dumps({'trainable_params':sum(p.numel() for p in m.parameters() if p.requires_grad),
   'batch_shape':list(tensor[0].shape),'loss':float(loss.detach()),'stage2_gradient':True,
   'stage3_gradient':True,'decoder_gradient':True,'stage1_gradient':False},indent=2))
  print('SMOKE PASS',tuple(tensor[0].shape),'loss',float(loss.detach()))
  return
 t.train()
 assert latest.exists() and torch.load(latest,map_location='cpu',weights_only=False)['epoch']==cfg.cfg['Global']['epoch_num']
 print('COMPLETE',latest)
if __name__=='__main__':main()
'''
S2_SCRIPT=S2_SCRIPT.replace("\'", "'")
SCRIPT=WORK/'s2_train_pair.py';SCRIPT.write_text(S2_SCRIPT)
for name,cfg,gpu in [('S2_NOAUG',NOAUG_CFG,'0'),('S2_AUG',AUG_CFG,'1')]:
 env={**os.environ,'CUDA_VISIBLE_DEVICES':gpu,'PYTHONPATH':str(OPENOCR)}
 proc=subprocess.run([sys.executable,str(SCRIPT),'--config',str(cfg),'--smoke'],cwd=OPENOCR,env=env,capture_output=True,text=True)
 print(name,'SMOKE',proc.returncode,(proc.stdout+proc.stderr)[-3500:])
 assert proc.returncode==0,name


## 6. Train 12 epoch trên hai T4 song song

Mỗi nhánh dùng một T4. Lúc session bị ngắt, `latest.pth` cho phép resume đúng epoch đã lưu; nếu Kaggle session biến mất, gắn output version trước vào input rồi chép checkpoint của đúng nhánh sang cùng thư mục output trước khi chạy lại. Không chọn `best.pth` theo IIIT5K test. Lưu log riêng cho hai nhánh.


In [ ]:
import time
jobs=[]
for name,cfg,gpu in [('S2_NOAUG',NOAUG_CFG,'0'),('S2_AUG',AUG_CFG,'1')]:
 env={**os.environ,'CUDA_VISIBLE_DEVICES':gpu,'PYTHONPATH':str(OPENOCR)}
 log=WORK/f'{name}_train.log';handle=open(log,'a')
 p=subprocess.Popen([sys.executable,str(SCRIPT),'--config',str(cfg)],cwd=OPENOCR,env=env,stdout=handle,stderr=subprocess.STDOUT)
 jobs.append((name,p,handle,log))
while any(p.poll() is None for _,p,_,_ in jobs):
 time.sleep(30)
 for name,p,h,log in jobs:
  if p.poll() is None:
   lines=log.read_text(errors='replace').splitlines()
   progress=next((x for x in reversed(lines) if 'epoch:' in x and 'global_step:' in x),'starting')
   print(name,progress[-180:])
for name,p,h,log in jobs:
 h.close();print(name,'exit',p.returncode)
 if p.returncode:print(log.read_text(errors='replace')[-6000:])
 assert p.returncode==0,name
 latest=WORK/name/'latest.pth';saved=torch.load(latest,map_location='cpu',weights_only=False)
 assert saved['epoch']==EXP['epochs'] and saved['global_step']>0
 print(name,'epoch',saved['epoch'],'updates',saved['global_step'],'sha',sha(latest))
assert torch.load(WORK/'S2_NOAUG/latest.pth',map_location='cpu',weights_only=False)['global_step']==torch.load(
 WORK/'S2_AUG/latest.pth',map_location='cpu',weights_only=False)['global_step'],'Different training budgets'


## 7. Đánh giá công bằng và kiểm từng ảnh

Đo B0, `S2_NOAUG` và `S2_AUG` trên cùng LMDB clean/motion mild/medium/severe. Dùng **checkpoint cuối epoch 12**. Xuất cả từng dự đoán và số ảnh một nhánh sửa đúng/làm sai so với nhánh kia. Nếu baseline không khớp notebook bạn (98,60% sạch / 80,23% blur nặng), dừng kết luận ngang notebook bạn và kiểm data/config.


In [ ]:
NAME_MODELS={'B0_official':(SV_CFG,SV_CKPT),
 'S2_NOAUG':(str(NOAUG_CFG),str(WORK/'S2_NOAUG/latest.pth')),
 'S2_AUG':(str(AUG_CFG),str(WORK/'S2_AUG/latest.pth'))}
SCORES={name:eval_all(name,cfg,ckpt,'svtrv2') for name,(cfg,ckpt) in NAME_MODELS.items()}
results=pd.DataFrame(SCORES).T[CONDS];display(results.round(2))
results.to_csv(WORK/'S2_AUG_FINAL_COMPARISON.csv')
baseline_ok=abs(results.loc['B0_official','clean']-98.6)<.1 and abs(results.loc['B0_official','motion_severe']-80.23)<.1
print('Matches friend baseline within 0.1 pp?',baseline_ok)
if not baseline_ok:print('Protocol differs; do not compare directly with friend 89.70% result.')
ax=results.T.plot.bar(figsize=(10,4));ax.set_ylabel('Word accuracy (%)');ax.set_ylim(70,100)
plt.tight_layout();plt.savefig(WORK/'S2_AUG_results.png',dpi=150);plt.show()


In [ ]:
PS_SCRIPT='\nimport os, sys, json\n__dir__ = os.path.dirname(os.path.abspath(__file__))\nsys.path.append(__dir__)\nsys.path.insert(0, os.path.abspath(os.path.join(__dir__, \'..\')))\nfrom tools.engine.config import Config\nfrom tools.engine.trainer import Trainer\nfrom tools.utility import ArgsParser\n\n\ndef norm(m, s):\n    if m.ignore_space: s = s.replace(\' \', \'\')\n    if m.is_filter: s = m._normalize_text(s)\n    if m.is_lower: s = s.lower()\n    return s\n\n\ndef main():\n    FLAGS = ArgsParser().parse_args()\n    cfg = Config(FLAGS.config)\n    FLAGS = vars(FLAGS)\n    opt = FLAGS.pop(\'opt\')\n    cfg.merge_dict(FLAGS)\n    cfg.merge_dict(opt)\n    trainer = Trainer(cfg, mode=\'eval\')\n    m = trainer.eval_class\n    inners = {\'self\': m} if type(m).__name__ == \'RecMetric\' else {k: v for k, v in vars(m).items() if type(v).__name__ == \'RecMetric\'}\n    assert inners, \'không thấy RecMetric trong \' + type(m).__name__\n    recs = {k: [] for k in inners}\n    def mk(k, orig):\n        def wrapped(pred_label, *a, **kw):\n            preds, labels = pred_label\n            for (p, _), (t, _) in zip(preds, labels): recs[k].append((p, t))\n            return orig(pred_label, *a, **kw)\n        return wrapped\n    for k, im in inners.items():\n        assert not getattr(im, \'with_ratio\', False) and not getattr(im, \'stream\', False), k\n        im.eval_metric = mk(k, im.eval_metric)\n    metric = trainer.eval()\n    best = None\n    for k in sorted(inners, key=lambda k: 0 if k in (\'self\', \'ctc_metric\', \'final_metric\') else 1):   # nhánh CTC/final là nhánh cho \'acc\' chính\n        if not recs[k]: continue\n        c_ = [norm(inners[k], p) == norm(inners[k], t) for p, t in recs[k]]\n        if abs(sum(c_) / len(c_) - metric[\'acc\']) < 1e-4: best = (k, recs[k], c_); break\n    if not best:\n        cand = [(abs(sum(norm(inners[k], p) == norm(inners[k], t) for p, t in recs[k]) / len(recs[k]) - metric[\'acc\']), k) for k in inners if recs[k]]\n        assert cand, \'không ghi được dự đoán từng mẫu\'\n        d_, k_ = min(cand); assert d_ < 0.01, f"không nhánh metric nào khớp acc chính {metric[\'acc\']}: lệch gần nhất {d_:.4f}"\n        print(\'CẢNH BÁO: chọn nhánh\', k_, \'lệch\', round(d_, 4)); best = (k_, recs[k_], [norm(inners[k_], p) == norm(inners[k_], t) for p, t in recs[k_]])\n    k, rec, correct = best; acc = sum(correct) / len(correct)\n    json.dump({\'acc\': acc, \'n\': len(rec), \'targets\': [t for _, t in rec], \'preds\': [p for p, _ in rec], \'correct\': correct},\n              open(os.environ[\'PERSAMPLE_OUT\'], \'w\'), ensure_ascii=False)\n\n\nif __name__ == \'__main__\':\n    main()\n'
(OPENOCR/'tools/eval_persample.py').write_text(PS_SCRIPT)
def persample(name,cond):
 cfg,ckpt=NAME_MODELS[name];dest=WORK/'persample_s2'/name/f'{cond}.json'
 if dest.exists():return json.loads(dest.read_text())
 dest.parent.mkdir(parents=True,exist_ok=True)
 key=f'Eval.dataset.data_dir_list=[{LM/cond}/test]'
 cmd=[sys.executable,'tools/eval_persample.py','--c',cfg,'--o',f'Global.pretrained_model={ckpt}',key,
      'Eval.loader.batch_size_per_card=128','Global.use_amp=False']
 run=subprocess.run(cmd,cwd=OPENOCR,env={**os.environ,'PERSAMPLE_OUT':str(dest)},capture_output=True,text=True)
 assert run.returncode==0 and dest.exists(),(name,cond,(run.stdout+run.stderr)[-2000:])
 return json.loads(dest.read_text())
PRED={name:{cond:persample(name,cond) for cond in CONDS} for name in NAME_MODELS}
for name in NAME_MODELS:
 for cond in CONDS:
  r=PRED[name][cond]
  assert r['n']==3000 and abs(100*r['acc']-results.loc[name,cond])<.05
for cond in CONDS:
 assert PRED['B0_official'][cond]['targets']==PRED['S2_NOAUG'][cond]['targets']==PRED['S2_AUG'][cond]['targets']
print('PER-SAMPLE AUDIT PASS: all 3 models, 4 conditions, 3000 aligned labels')


In [ ]:
rng=np.random.default_rng(20261115);rows=[]
for ref,cand in [('B0_official','S2_NOAUG'),('B0_official','S2_AUG'),('S2_NOAUG','S2_AUG')]:
 for cond in CONDS:
  a=np.array(PRED[ref][cond]['correct'],bool);b=np.array(PRED[cand][cond]['correct'],bool)
  assert len(a)==len(b)==3000
  diff=b.astype(np.int8)-a.astype(np.int8)
  ix=rng.integers(0,len(diff),size=(2000,len(diff)))
  low,high=np.quantile(diff[ix].mean(1)*100,[.025,.975])
  rows.append(dict(reference=ref,candidate=cand,condition=cond,fixed=int((~a&b).sum()),
   broken=int((a&~b).sum()),delta_pp=float(diff.mean()*100),low_95_pp=float(low),high_95_pp=float(high)))
PAIRED=pd.DataFrame(rows);PAIRED.to_csv(WORK/'S2_AUG_PAIRED.csv',index=False);display(PAIRED)
print('Primary claim = S2_AUG versus S2_NOAUG, especially motion_severe; not versus untrained B0 alone.')


## 8. Gói gửi để audit

Gói nhẹ chứa bảng kết quả, dự đoán từng ảnh, config, augmentation source, log và hash. Không chứa checkpoint vài chục MB; nếu cần chạy lại mô hình phải lưu riêng output Kaggle. Báo cáo phải phân biệt kết quả của **S2/RCTC** ở notebook này với **GTC+RCTC train toàn mô hình** trong notebook bạn.


In [ ]:
import zipfile
TRAIN_AUDIT={}
for name in ('S2_NOAUG','S2_AUG'):
 ck=WORK/name/'latest.pth';meta=torch.load(ck,map_location='cpu',weights_only=False)
 TRAIN_AUDIT[name]=dict(epoch=meta['epoch'],global_step=meta['global_step'],
   checkpoint_sha=sha(ck),smoke=json.loads((WORK/name/'smoke.json').read_text()))
AUDIT=dict(openocr_commit=subprocess.check_output(['git','-C',str(OPENOCR),'rev-parse','HEAD'],text=True).strip(),
 source_friend='aug-svtrv2.ipynb',source_friend_sha='dde0aa93d38f47579713451979f9f1a74dc3cb5873658209bcedc62be63ffce7',
 official_checkpoint_sha=sha(CK),official_config_sha=sha(SV_CFG),
 converted_s2_sha=sha(S2_INIT),train_dirs=TRAIN_DIRS,train_lmdb_sha=[sha(Path(x)/'data.mdb') for x in TRAIN_DIRS],
 blur_manifest_sha=sha(man_path),baseline_matches_friend=bool(baseline_ok),
 augmentation=AUG_OP,motion_aug_sha=sha(OPENOCR/'openrec/preprocess/motion_blur_aug.py'),
 training=TRAIN_AUDIT,results=results.to_dict(),limitations=['test manifest used to set blur range',
 'MotionBlurAug from supplied notebook seeds RNG from worker PID/time; exact replay is not guaranteed',
 'Official CTC label/loss replaces canonicalized pilot S2 loss; compare AUG vs NOAUG internally',
 'IIIT5K test previously inspected','Union14M content overlap with IIIT5K not exhaustively checked'])
(WORK/'S2_AUG_AUDIT.json').write_text(json.dumps(AUDIT,indent=2,default=float))
package=Path('/kaggle/working/Danh_S2_Union14M_MotionAug_audit_only.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
 for p in [WORK/'S2_AUG_FINAL_COMPARISON.csv',WORK/'S2_AUG_PAIRED.csv',WORK/'S2_AUG_AUDIT.json',
  WORK/'S2_AUG_results.png',WORK/'S2_AUG_pipeline.png',WORK/'data_phase.png',NOAUG_CFG,AUG_CFG,SCRIPT,WORK/'S2_NOAUG_train.log',WORK/'S2_AUG_train.log']:
  assert p.exists(),p;z.write(p,p.name)
 for name in ('S2_NOAUG','S2_AUG'):
  z.write(WORK/name/'smoke.json',f'{name}/smoke.json')
 for p in (WORK/'persample_s2').rglob('*.json'):z.write(p,str(p.relative_to(WORK)))
print('Send this file:',package,'MB',round(package.stat().st_size/1e6,2))
